# DATA ANALYST TEST
**Author:** Ashkan Moradi
**Date:** November 2025

Hi. let's Import our libraries:

In [ ]:
import pandas as pd

Now let's load our csv files into dataframes:

In [ ]:
orders = pd.read_csv("orders.csv")
order_items = pd.read_csv("order_items.csv")
products = pd.read_csv("products.csv")

Now let's do some Exploratory Data Analysis (EDA) and take a brief look into our data.
We take a look into five first rows, column types, and Number of rows and columns in each table.

Let's start with order table:

In [ ]:
# orders table:
print("First five rows:") 
print(orders.head(5))

print("\n Data Types:")
print(orders.dtypes)

print("\n Number of rows and columns:") 
print(orders.shape)

Now Let's take a look at order_items table:

In [ ]:
# order_items table:
print("First five rows:") 
print(order_items.head(5))

print("\n Data Types:")
print(order_items.dtypes)

print("\n Number of rows and columns:") 
print(order_items.shape)

And now Let's take a look at products table:

In [ ]:
# products table:
print("First five rows:") 
print(products.head(5))

print("\n Data Types:")
print(products.dtypes)

print("\n Number of rows and columns:") 
print(products.shape)

Now We start with Data Quality Checks (part a)

In [ ]:
# orders duplicates/null/integrity
duplicate_orderIdAndCustomerId = orders.duplicated(subset=['order_id','customer_id'],keep=False)
print(orders[duplicate_orderIdAndCustomerId])

print(orders.isnull().sum())

missing_orders = orders[~orders['order_id'].isin(order_items['order_id'])]
print(missing_orders)

In [ ]:
# order_items data quality
duplicate_productIdAndOrderId = order_items.duplicated(subset=['order_id', 'product_id'],keep=False)
print(order_items[duplicate_productIdAndOrderId])

print(order_items.isnull().sum())

missing_orders = order_items[~order_items['order_id'].isin(orders['order_id'])]
print(missing_orders)

missing_products = order_items[~order_items['product_id'].isin(products['product_id'])]
print(missing_products)

In [ ]:
# products data quality
duplicate_productId = products.duplicated(subset=['product_id'],keep=False)
print(products[duplicate_productId])

print(products.isnull().sum())

missing_products = products[~products['product_id'].isin(order_items['product_id'])]
print(missing_products)

Now Data Cleaning

In [ ]:
order_items = order_items.drop_duplicates(subset=['order_id', 'product_id'],keep='first')

products.drop(products[products['product_id'] == -1].index, inplace=True)

products['unit_cost'] = products['unit_cost'].str.removeprefix('$')
products['unit_cost'] = products['unit_cost'].astype('float64')

order_items['sales'] = order_items['quantity'] * order_items['unit_price']

Sales Metrics

In [ ]:
order_items_merged = order_items.merge(products, on="product_id", how="left")

order_items_merged['gross_revenue'] = order_items_merged['quantity'] * order_items_merged['unit_price']
order_items_merged['net_revenue'] = order_items_merged['gross_revenue'] - order_items_merged['discounts']
order_items_merged['cost'] = order_items_merged['unit_cost'] * order_items_merged['quantity']

category_metrics = order_items_merged.groupby('category').agg(
    total_gross_revenue=('gross_revenue', 'sum'),
    total_net_revenue=('net_revenue', 'sum'),
    total_cost=('cost', 'sum')
).reset_index()

category_metrics

Margins

In [ ]:
category_metrics['gross_margin_perc'] = (category_metrics['total_gross_revenue'] - category_metrics['total_cost']) / category_metrics['total_gross_revenue'] * 100
category_metrics['net_margin_perc'] = (category_metrics['total_net_revenue'] - category_metrics['total_cost']) / category_metrics['total_net_revenue'] * 100

category_metrics

Customer Metrics

In [ ]:
customer_spending = orders.groupby('customer_id').agg(total_spent=('total_amount', 'sum')).reset_index()

top_customers = customer_spending.sort_values(by='total_spent', ascending=False).head(2)

top_customers

In [ ]:
customer_avg_order = orders.groupby('customer_id').agg(avg_order_value=('total_amount', 'mean')).reset_index()

customer_avg_order

Frequently Purchased Products

In [ ]:
from collections import Counter
from itertools import combinations

order_product_groups = order_items.groupby('order_id')['product_id'].apply(list)

pair_counter = Counter()

for products_list in order_product_groups:
    pairs = combinations(sorted(products_list), 2)
    pair_counter.update(pairs)

top_3_pairs = pair_counter.most_common(3)
top_3_pairs